B16. Масштабирование панели 3→10 участников: прямая проверка предела k_eff.....


In [1]:
import sys, os
from itertools import combinations

def _build_data_tools():
    import json
    import os
    import re
    import warnings
    import numpy as np
    import pandas as pd
    from scipy import stats
    warnings.filterwarnings('ignore')

    def path_join(*parts):
        return os.path.normpath(os.path.join(*parts))

    def normalize_newlines(frame):
        for column in frame.select_dtypes(include=['object', 'string']).columns:
            frame[column] = frame[column].map(lambda value: value.replace('\r\n', '\n').replace('\r', '\n') if isinstance(value, str) else value)
        return frame
    from pathlib import Path
    _start = Path.cwd().resolve()
    _repo = next((p for p in (_start, *_start.parents) if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()), None)
    if _repo is None:
        raise FileNotFoundError('Run the notebook from inside the repository.')
    if os.name == 'nt' and (not str(_repo).startswith(chr(92) * 2 + '?' + chr(92))):
        _repo = Path(chr(92) * 2 + '?' + chr(92) + str(_repo))
    ROOT = str(_repo)
    HERE = str(_repo / 'analyses/protocol_design')
    OUT = path_join(HERE, 'results')
    os.makedirs(OUT, exist_ok=True)
    PBASE_DIR = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete'
    HOMOG_DS_DIR = 'experiments/homogeneous_panels/three_deepseekv4_members_complete'
    NB = 'experiments/clinical_protocols/single_model_baselines/baseline_%s/llm_as_judge/baseline_%s_%s_judge.csv'
    USC = 'experiments/token_budget_controls/universal_self_consistency_gpt_seven_samples/llm_as_judge/usc_gpt_sc%d_judge.csv'
    HOMOG_DIRS = {'gpt': 'experiments/homogeneous_panels/three_gpt_members', 'qwen': 'experiments/homogeneous_panels/three_qwen_members', 'gemini': 'experiments/homogeneous_panels/three_gemini_members'}
    PANELS_OTHER_DIR = {'P-weak': 'experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro', 'P-mixed': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm', 'P-strong': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120'}
    PANEL_FILE = {'P-weak': 'delphi_pweak_noroles_judge.csv', 'P-mixed': 'delphi_pmixed_noroles_judge.csv', 'P-strong': 'delphi_pstrong_noroles_judge.csv'}
    SINGLES = ['gpt', 'qwen', 'deepseekv4', 'deepseekv4pro', 'gemini', 'glm', 'kimi']
    COMPOSITION = {'P-base': ['gpt', 'qwen', 'deepseekv4'], 'P-weak': ['qwen', 'gemini', 'deepseekv4pro'], 'P-mixed': ['deepseekv4', 'gemini', 'glm'], 'P-strong': ['deepseekv4', 'glm', 'gpt'], 'homog gpt': ['gpt'] * 3, 'homog qwen': ['qwen'] * 3, 'homog deepseekv4': ['deepseekv4'] * 3, 'homog gemini': ['gemini'] * 3}
    HETERO = {'P-base': 1, 'P-weak': 1, 'P-mixed': 1, 'P-strong': 1, 'homog gpt': 0, 'homog qwen': 0, 'homog deepseekv4': 0, 'homog gemini': 0}
    PANELS = list(COMPOSITION)
    LADDER = ['baseline gpt', 'USC@3 gpt', 'USC@5 gpt', 'USC@7 gpt', 'homog gpt', 'homog deepseekv4', 'P-base']

    def arm_paths(design='noroles'):
        p = {}
        for m in SINGLES:
            p['baseline ' + m] = NB % (m, design, m)
        if design == 'noroles':
            for k in (3, 5, 7):
                p['USC@%d gpt' % k] = USC % k
            for tag, d in HOMOG_DIRS.items():
                p['homog ' + tag] = d + '/llm_as_judge/delphi_homog_%s_noroles_judge.csv' % tag
            p['homog deepseekv4'] = HOMOG_DS_DIR + '/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv'
            for k, d in PANELS_OTHER_DIR.items():
                p[k] = d + '/llm_as_judge/' + PANEL_FILE[k]
        p['P-base'] = PBASE_DIR + '/llm_as_judge/delphi_base_%s_judge.csv' % design
        return p
    ROUNDS_DIR = {'P-base': (PBASE_DIR, 'delphi_base_noroles_protocol_r%d_judge.csv'), 'P-weak': (PANELS_OTHER_DIR['P-weak'], 'delphi_pweak_noroles_protocol_r%d_judge.csv'), 'P-mixed': (PANELS_OTHER_DIR['P-mixed'], 'delphi_pmixed_noroles_protocol_r%d_judge.csv'), 'P-strong': (PANELS_OTHER_DIR['P-strong'], 'delphi_pstrong_noroles_protocol_r%d_judge.csv'), 'homog gpt': (HOMOG_DIRS['gpt'], 'delphi_homog_gpt_noroles_protocol_r%d_judge.csv'), 'homog gemini': (HOMOG_DIRS['gemini'], 'delphi_homog_gemini_noroles_protocol_r%d_judge.csv'), 'homog deepseekv4': (HOMOG_DS_DIR, 'delphi_homog_deepseekv4_noroles_protocol_r%d_judge.csv')}
    RETEST = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_retest_r%d_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_retest_r%d_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_retest_r%d_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_retest_r%d_judge.csv'}
    SHORT = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_short_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_short_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_short_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_short_judge.csv'}
    JUDGE41 = {'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_gpt41/baseline_%s_gpt_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_gpt41/baseline_%s_qwen_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_gpt41/baseline_%s_deepseekv4_judge.csv'}
    CASE_COVARS = ['Type', 'Sub Clinical Specialties', 'Age Group', 'Genealogy', 'Diagnosis10', 'Vignette_race']
    DEXP = 'experiments/panel_parameter_sweeps'
    SWEEPS = {'experts': (DEXP + '/experts/llm_as_judge', 'delphi_experts_%s_judge.csv'), 'questionnaire_items': (DEXP + '/questionnaire_items/llm_as_judge', 'delphi_questionnaire_items_%s_judge.csv'), 'questions_per_expert': (DEXP + '/questions_per_expert/llm_as_judge', 'delphi_questions_per_expert_%s_judge.csv'), 'temperature': (DEXP + '/temperature/llm_as_judge', 'delphi_temperature_%s_judge.csv'), 'rounds': (DEXP + '/rounds/llm_as_judge', 'delphi_%s_judge.csv')}
    EXPERT_GROUPS = {'group1_10models': 10, 'group2_4models': 4, 'group3_5models': 5, 'group4_5models': 5, 'group5_7models': 7, 'group6_5models': 5, 'group7_6models': 6}
    EXT = 'experiments'
    MEDMCQA = {'baseline deepseekv4': EXT + '/external_benchmarks/medmcqa/baseline_deepseekv4/baseline_medmcqa_deepseekv4.csv', 'baseline gpt': EXT + '/external_benchmarks/medmcqa/baseline_gpt/baseline_medmcqa_gpt.csv', 'baseline qwen': EXT + '/external_benchmarks/medmcqa/baseline_qwen/baseline_medmcqa_qwen.csv', 'baseline alice': EXT + '/external_benchmarks/initial_runs/baseline_alice/baseline_medmcqa_alice.csv', 'baseline qwen36': EXT + '/external_benchmarks/initial_runs/baseline_qwen36/baseline_medmcqa_qwen36.csv', 'P-ac (gpt/qwen/ds)': EXT + '/external_benchmarks/medmcqa/panel_gpt120_organizer_qwen_deepseek/delphi_medmcqa_noroles.csv', 'P-ac (gpt/qwen/alice)': EXT + '/external_benchmarks/initial_runs/panel_gpt120_organizer_qwen_alice/delphi_medmcqa_noroles.csv', 'P-deepseektrio': EXT + '/external_benchmarks/initial_runs/three_deepseek_members/delphi_medmcqa_noroles.csv', 'P-base-like': EXT + '/external_benchmarks/initial_runs/heterogeneous_gpt120_organizer/delphi_medmcqa_noroles.csv'}

    def rd_any(rel, index=None):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        with open(p, encoding='utf-8-sig', errors='replace') as fh:
            head = fh.readline()
        sep = ';' if head.count(';') > head.count(',') else ','
        d = normalize_newlines(pd.read_csv(p, sep=sep, encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if index and index in d.columns:
            d = d.dropna(subset=[index]).drop_duplicates(index).set_index(index)
        return d

    def sweep_arms(kind, verbose=True):
        import glob as _g
        folder, _ = SWEEPS[kind]
        out = {}
        for p in sorted(_g.glob(path_join(ROOT, folder, '*_judge.csv'))):
            name = os.path.basename(p)[:-len('_judge.csv')]
            d = rd(os.path.relpath(p, ROOT).replace('\\', '/'))
            if d is not None:
                out[name] = d
        if verbose:
            print('свип %s: плеч %d, кейсов %d' % (kind, len(out), len(next(iter(out.values()))) if out else 0))
        return out

    def member_triage_acc(d):
        t = true_label(d)
        out = {}
        for p in participants(d):
            lab = d['r1_emergency__' + p].map(norm_label)
            ok = lab.notna() & t.notna()
            out[p] = float((lab[ok] == t[ok]).mean()) if ok.any() else np.nan
        return pd.Series(out)

    def rd(rel, need_score=True):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        d = normalize_newlines(pd.read_csv(p, sep=';', encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if need_score and 'Judge_score' not in d.columns:
            return None
        sub = ['Judge_score', 'Vignette'] if need_score else ['Vignette']
        d = d.dropna(subset=sub).drop_duplicates('Vignette', keep='first')
        return d.set_index('Vignette')

    def load_arms(design='noroles', verbose=True):
        a = {}
        for name, rel in arm_paths(design).items():
            d = rd(rel)
            if d is not None:
                a[name] = d
        if verbose:
            print('плеч загружено: %d' % len(a))
            print('   ' + ', '.join(a))
        return a

    def js(arms, name):
        return pd.to_numeric(arms[name]['Judge_score'], errors='coerce').dropna()

    def score_matrix(arms, names=None):
        names = list(arms) if names is None else names
        m = pd.concat([js(arms, n).rename(n) for n in names], axis=1)
        return m.dropna()

    def meta(arms=None):
        d = rd(PBASE_DIR + '/llm_as_judge/delphi_base_noroles_judge.csv')
        cols = [c for c in CASE_COVARS if c in d.columns]
        return d[cols].copy()

    def item_frame(d):
        r = {}
        if d is None or 'judge_details' not in d.columns:
            return r
        for v, s in d['judge_details'].dropna().items():
            try:
                arr = json.loads(s)
            except Exception:
                continue
            for it in arr:
                ax = next((t.split(':', 1)[1] for t in it.get('tags') or [] if isinstance(t, str) and t.startswith('axis:')), None)
                r[v, it['id']] = {'met': bool(it.get('met')), 'points': it.get('points', 0) or 0, 'axis': ax}
        return r

    def load_items(design='noroles', arms_wanted=None, verbose=True):
        paths = arm_paths(design)
        if arms_wanted:
            paths = {k: v for k, v in paths.items() if k in arms_wanted}
        items = {}
        for name, rel in paths.items():
            d = rd(rel)
            f = item_frame(d)
            if f:
                items[name] = f
        common = None
        for f in items.values():
            common = set(f) if common is None else common & set(f)
        ref = next(iter(items.values()))
        pos = sorted([k for k in common or set() if ref[k]['points'] > 0])
        if verbose:
            print('плеч с попунктными вердиктами: %d' % len(items))
            print('общих положительных пунктов: %d' % len(pos))
        return (items, pos)

    def items_long(items, pos, arms_wanted=None):
        names = arms_wanted or list(items)
        rows = []
        for a in names:
            f = items[a]
            for k in pos:
                it = f[k]
                rows.append({'case': k[0], 'cid': k[1], 'arm': a, 'met': int(it['met']), 'points': it['points'], 'axis': it['axis']})
        return pd.DataFrame(rows)

    def paired(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(float), b.loc[ix].astype(float))
        d = x - y
        p = stats.wilcoxon(x, y)[1] if (d != 0).any() else np.nan
        t = stats.ttest_rel(x, y)
        return {'n': len(ix), 'delta': float(d.mean()), 'sd': float(d.std(ddof=1)), 'share_gt': float((d > 0).mean()), 'p_wilcoxon': float(p), 't': float(t.statistic), 'p_ttest': float(t.pvalue)}

    def se_clt(x):
        x = np.asarray(x, float)
        n = len(x)
        return float(np.sqrt(x.var(ddof=1) / n))

    def se_clustered(x, cluster, cr1=True):
        x = np.asarray(x, float)
        cl = np.asarray(cluster)
        n = len(x)
        e = x - x.mean()
        df = pd.DataFrame({'e': e, 'c': cl})
        r = df.groupby('c', observed=True)['e'].sum().to_numpy()
        ss = df.groupby('c', observed=True)['e'].apply(lambda s: float((s ** 2).sum())).to_numpy()
        cross = float((r ** 2 - ss).sum())
        v = se_clt(x) ** 2 + cross / n ** 2
        G = len(r)
        if cr1 and G > 1:
            v *= G / (G - 1.0)
        return (float(np.sqrt(max(v, 0.0))), G)

    def cluster_test(d, cluster, cr1=True):
        d = pd.Series(d).astype(float)
        cl = pd.Series(cluster).reindex(d.index)
        ok = cl.notna()
        d, cl = (d[ok], cl[ok])
        m = float(d.mean())
        s0 = se_clt(d)
        s1, G = se_clustered(d, cl, cr1=cr1)
        dfree = max(G - 1, 1)
        t = m / s1 if s1 > 0 else np.nan
        p = float(2 * stats.t.sf(abs(t), dfree)) if np.isfinite(t) else np.nan
        tc = stats.t.ppf(0.975, dfree)
        return {'n': len(d), 'G': G, 'mean': m, 'se_naive': s0, 'se_cluster': s1, 'DEFF': (s1 / s0) ** 2 if s0 > 0 else np.nan, 't': float(t), 'df': dfree, 'p': p, 'ci_lo': m - tc * s1, 'ci_hi': m + tc * s1, 'p_naive': float(2 * stats.norm.sf(abs(m / s0))) if s0 > 0 else np.nan}

    def icc_anova(values, groups):
        df = pd.DataFrame({'y': np.asarray(values, float), 'g': np.asarray(groups)})
        df = df.dropna()
        grp = df.groupby('g', observed=True)['y']
        ni = grp.size().to_numpy().astype(float)
        k = len(ni)
        N = ni.sum()
        if k < 2 or N <= k:
            return (0.0, float(ni.mean() if k else 1), k)
        gm = df['y'].mean()
        msb = float((ni * (grp.mean().to_numpy() - gm) ** 2).sum() / (k - 1))
        ssw = float(((df['y'] - df['g'].map(grp.mean()).to_numpy()) ** 2).sum())
        msw = ssw / (N - k)
        B0 = float((N - (ni ** 2).sum() / N) / (k - 1))
        rho = (msb - msw) / (msb + (B0 - 1) * msw) if msb + (B0 - 1) * msw > 0 else 0.0
        return (float(np.clip(rho, 0, 1)), B0, k)

    def ci_wald(p, n, z=1.96):
        se = np.sqrt(p * (1 - p) / n)
        return (p - z * se, p + z * se)

    def ci_wilson(p, n, z=1.96):
        d = 1 + z ** 2 / n
        c = (p + z ** 2 / (2 * n)) / d
        h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
        return (c - h, c + h)

    def ci_agresti_coull(x, n, z=1.96):
        nt = n + z ** 2
        pt = (x + z ** 2 / 2) / nt
        se = np.sqrt(pt * (1 - pt) / nt)
        return (pt - z * se, pt + z * se)

    def ci_clopper_pearson(x, n, alpha=0.05):
        lo = 0.0 if x == 0 else stats.beta.ppf(alpha / 2, x, n - x + 1)
        hi = 1.0 if x == n else stats.beta.ppf(1 - alpha / 2, x + 1, n - x)
        return (float(lo), float(hi))

    def boot_ci(x, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        x = np.asarray(x, float)
        n = len(x)
        s = np.array([stat(x[rng.integers(0, n, n)]) for _ in range(B)])
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def boot_ci_cluster(x, cluster, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        df = pd.DataFrame({'x': np.asarray(x, float), 'c': np.asarray(cluster)})
        groups = [g['x'].to_numpy() for _, g in df.groupby('c', observed=True)]
        G = len(groups)
        s = []
        for _ in range(B):
            pick = rng.integers(0, G, G)
            s.append(stat(np.concatenate([groups[i] for i in pick])))
        s = np.array(s)
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def holm(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 0.0
        for i, k in enumerate(order):
            run = max(run, (m - i) * p[k])
            adj[k] = min(run, 1.0)
        return adj

    def bh(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 1.0
        for i in range(m - 1, -1, -1):
            k = order[i]
            run = min(run, m * p[k] / (i + 1))
            adj[k] = min(run, 1.0)
        return adj

    def bonferroni(pvals):
        p = np.asarray(pvals, float)
        return np.minimum(p * len(p), 1.0)

    def mcnemar(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(bool), b.loc[ix].astype(bool))
        n01, n10 = (int((~x & y).sum()), int((x & ~y).sum()))
        if n01 + n10 == 0:
            return {'n01': n01, 'n10': n10, 'p': 1.0}
        return {'n01': n01, 'n10': n10, 'p': float(stats.binomtest(n10, n01 + n10, 0.5).pvalue)}
    ORD = {'SELF': 0, 'NONEMERG': 1, 'EMERG': 2}
    GT = {'Self-care appropriate': 'SELF', 'Requires non-emergent care': 'NONEMERG', 'Requires emergent care': 'EMERG'}
    _DASH = dict.fromkeys(map(ord, '‐‑‒–—―−\xad'), '-')
    _KEYS = [('requires emergent care', 'EMERG'), ('requires non-emergent care', 'NONEMERG'), ('self-care appropriate', 'SELF')]

    def norm_label(x):
        if not isinstance(x, str) or not x.strip():
            return None
        if x.strip() in GT:
            return GT[x.strip()]
        import unicodedata
        s = unicodedata.normalize('NFKC', x).translate(_DASH).lower()
        s = re.sub('\\s+', ' ', re.sub('[^a-z\\- ]', ' ', s)).strip()
        for k, v in _KEYS:
            if s.startswith(k) or k in s:
                return v
        if 'non-emergent' in s or 'non emergent' in s:
            return 'NONEMERG'
        if 'emergent' in s or 'emergency' in s:
            return 'EMERG'
        if 'self' in s:
            return 'SELF'
        return None
    _PAT = re.compile('emergency\\s*type\\s*[:\\-]\\s*[\\*_`#\\s]*([^\\n\\r\\*_|]{0,90})', re.I)

    def parse_triage(text):
        if not isinstance(text, str) or not text.strip():
            return None
        import unicodedata
        m = _PAT.search(unicodedata.normalize('NFKC', text).translate(_DASH))
        if not m:
            return None
        return norm_label('Requires ' + m.group(1) if False else m.group(1))

    def pred_triage(d):
        if 'delphi_consensus_emergency_type' in d.columns:
            s = d['delphi_consensus_emergency_type'].map(norm_label)
            if s.notna().mean() > 0.5:
                return s
        for c in ('winner_protocol', 'baseline_protocol_anonymous'):
            if c in d.columns:
                s = d[c].map(parse_triage)
                if s.notna().mean() > 0.3:
                    return s
        return pd.Series(index=d.index, dtype=object)

    def participants(d):
        return sorted((c.split('__', 1)[1] for c in d.columns if c.startswith('r1_emergency__')))

    def true_label(d):
        return d['Type'].map(GT)

    def words(s):
        return len(re.findall('\\S+', str(s))) if isinstance(s, str) else np.nan

    def fmt_p(p):
        if p is None or (isinstance(p, float) and (not np.isfinite(p))):
            return '  n/a '
        return '%.3f' % p if p >= 0.001 else '%.1e' % p

    def show(df, title=None, floatfmt='%.4f'):
        if title:
            print('\n' + title)
            print('-' * max(len(title), 60))
        with pd.option_context('display.width', 200, 'display.max_columns', 60, 'display.float_format', lambda v: floatfmt % v):
            print(df)

    def save(df, name):
        p = path_join(OUT, name)
        df.to_csv(p, sep=';', encoding='utf-8-sig')
        print('сохранено: out/%s  (%d строк)' % (name, len(df)))
        return p
    return locals()
globals().update({k: v for k, v in _build_data_tools().items() if not k.startswith('_')})
del _build_data_tools
import statsmodels.api as sm
EX = sweep_arms('experts')
RND = sweep_arms('rounds')
print()
REF3 = RND.get('delphi_optimum_early_stop')
print('референс k=3 (delphi_optimum_early_stop):', 'участников =', len(participants(REF3)), ' Judge = %.4f' % pd.to_numeric(REF3['Judge_score'], errors='coerce').mean())


свип experts: плеч 7, кейсов 60


свип rounds: плеч 2, кейсов 60

референс k=3 (delphi_optimum_early_stop): участников = 3  Judge = 0.8194


In [2]:
def group_row(name, d, k_override=None):
    js_ = pd.to_numeric(d['Judge_score'], errors='coerce')
    ps = participants(d)
    acc = member_triage_acc(d)
    t = true_label(d)
    pr = pred_triage(d)
    ok = pr.notna() & t.notna()
    return {'группа': name, 'k': k_override or len(ps), 'Judge': float(js_.mean()), 'SD': float(js_.std(ddof=1)), 'точность триажа панели': float((pr[ok] == t[ok]).mean()) if ok.any() else np.nan, 'ср. сила участников': float(acc.mean()), 'мин. участник': float(acc.min()), 'макс. участник': float(acc.max()), 'состав': ', '.join((p.replace('_default', '') for p in ps))}
rows = [group_row('k=3 (референс)', REF3)]
for name, d in EX.items():
    rows.append(group_row(name.replace('_judge', ''), d))
G = pd.DataFrame(rows).sort_values('k').set_index('группа')
show(G[['k', 'Judge', 'SD', 'точность триажа панели', 'ср. сила участников', 'мин. участник', 'макс. участник']], 'Группы разного размера: качество и сила состава')
print()
for i, r in G.iterrows():
    print('  %-22s k=%2d  %s' % (i, r['k'], r['состав']))



Группы разного размера: качество и сила состава
------------------------------------------------------------
                                 k  Judge     SD  точность триажа панели  ср. сила участников  мин. участник  макс. участник
группа                                                                                                                      
k=3 (референс)                   3 0.8194 0.1146                  0.8167               0.7677         0.6833          0.8364
delphi_experts_group2_4models    4 0.8071 0.1187                  0.7833               0.7694         0.6500          0.8644
delphi_experts_group3_5models    5 0.8276 0.1203                  0.7833               0.7567         0.7000          0.8500
delphi_experts_group4_5models    5 0.8169 0.1189                  0.8000               0.7840         0.7000          0.8500
delphi_experts_group6_5models    5 0.6384 0.2520                  0.7333               0.7347         0.6949          0.7833
delphi_experts_

In [3]:
ALL = {'k=3 (референс)': REF3}
ALL.update({n.replace('_judge', ''): d for n, d in EX.items()})
S = pd.DataFrame({n: pd.to_numeric(d['Judge_score'], errors='coerce') for n, d in ALL.items()}).dropna()
print('общих кейсов у всех групп:', len(S))
KS = G['k'].to_dict()
curve = pd.DataFrame({'k': [KS[c] for c in S.columns], 'Judge': S.mean().to_numpy(), 'SE': [se_clt(S[c]) for c in S.columns]}, index=S.columns)
show(curve.sort_values('k'), 'Кривая размер → качество')
rows = []
ref = S['k=3 (референс)']
for c in S.columns:
    if c == 'k=3 (референс)':
        continue
    d = S[c] - ref
    rows.append({'группа': c, 'k': KS[c], 'Δ к k=3': float(d.mean()), 'SE': se_clt(d), 'p (Уилкоксон)': float(stats.wilcoxon(d)[1]) if (d != 0).any() else np.nan, 'CI низ': d.mean() - 1.96 * se_clt(d), 'CI верх': d.mean() + 1.96 * se_clt(d)})
CMP = pd.DataFrame(rows).sort_values('k').set_index('группа')
show(CMP, 'Каждая группа против референсной тройки (парно, 60 кейсов)')


общих кейсов у всех групп: 60

Кривая размер → качество
------------------------------------------------------------
                                 k  Judge     SE
k=3 (референс)                   3 0.8194 0.0148
delphi_experts_group2_4models    4 0.8071 0.0153
delphi_experts_group3_5models    5 0.8276 0.0155
delphi_experts_group4_5models    5 0.8169 0.0154
delphi_experts_group6_5models    5 0.6384 0.0325
delphi_experts_group7_6models    6 0.8249 0.0137
delphi_experts_group5_7models    7 0.8390 0.0141
delphi_experts_group1_10models  10 0.8198 0.0151



Каждая группа против референсной тройки (парно, 60 кейсов)
------------------------------------------------------------
                                 k  Δ к k=3     SE  p (Уилкоксон)  CI низ  CI верх
группа                                                                            
delphi_experts_group2_4models    4  -0.0123 0.0163         0.4909 -0.0442   0.0197
delphi_experts_group3_5models    5   0.0082 0.0144         0.3498 -0.0201   0.0365
delphi_experts_group4_5models    5  -0.0025 0.0142         0.9415 -0.0303   0.0254
delphi_experts_group6_5models    5  -0.1810 0.0294         0.0000 -0.2386  -0.1234
delphi_experts_group7_6models    6   0.0055 0.0139         0.6152 -0.0218   0.0329
delphi_experts_group5_7models    7   0.0197 0.0146         0.1733 -0.0090   0.0483
delphi_experts_group1_10models  10   0.0004 0.0153         0.8734 -0.0296   0.0305


In [4]:
def fit(df, label):
    X = sm.add_constant(df[['k', 'ср. сила участников']].astype(float))
    m = sm.OLS(df['Judge'].astype(float), X).fit()
    out = pd.DataFrame({'коэф': m.params, 'SE': m.bse, 't': m.tvalues, 'p': m.pvalues})
    out.insert(0, 'модель', label)
    return (out, m)
full, m1 = fit(G, 'все 8 групп')
noout, m2 = fit(G[~G.index.str.contains('group6')], 'без группы 6')
show(pd.concat([full, noout]), 'Judge ~ размер панели + сила состава')
print()
print('R² все группы = %.3f (n = %d),  без группы 6 = %.3f (n = %d)' % (m1.rsquared, int(m1.nobs), m2.rsquared, int(m2.nobs)))
print()
print('ЧИТАЕТСЯ ТАК: коэффициент при `k` — это средний прирост Judge_score')
print('на одного дополнительного участника при неизменной силе состава.')
print('Если он статистически неотличим от нуля и мал по величине, предсказание')
print('B9 о насыщении подтверждается.')



Judge ~ размер панели + сила состава
------------------------------------------------------------
                           модель    коэф     SE       t      p
const                 все 8 групп -1.1039 1.1995 -0.9203 0.3997
k                     все 8 групп  0.0102 0.0116  0.8788 0.4197
ср. сила участников   все 8 групп  2.4362 1.5564  1.5652 0.1783
const                без группы 6  1.2049 0.2131  5.6545 0.0048
k                    без группы 6  0.0001 0.0017  0.0667 0.9501
ср. сила участников  без группы 6 -0.5039 0.2743 -1.8371 0.1401

R² все группы = 0.347 (n = 8),  без группы 6 = 0.519 (n = 7)

ЧИТАЕТСЯ ТАК: коэффициент при `k` — это средний прирост Judge_score
на одного дополнительного участника при неизменной силе состава.
Если он статистически неотличим от нуля и мал по величине, предсказание
B9 о насыщении подтверждается.


In [5]:
def keff_group(d):
    t = true_label(d)
    ps = participants(d)
    M_ = {}
    for p in ps:
        lab = d['r1_emergency__' + p].map(norm_label)
        M_[p] = (lab == t).astype(float).where(lab.notna() & t.notna())
    V = pd.DataFrame(M_).dropna()
    if V.shape[1] < 2 or V.shape[0] < 10:
        return (np.nan, np.nan, V.shape)
    C = V.corr()
    rs = [C.iloc[i, j] for i, j in combinations(range(C.shape[0]), 2)]
    rbar = float(np.nanmean(rs))
    k = C.shape[0]
    return (rbar, k / (1 + (k - 1) * rbar), V.shape)
rows = []
for name, d in ALL.items():
    rbar, keff, shape = keff_group(d)
    rows.append({'группа': name, 'k': KS[name], 'ρ̄ (по ошибкам R1)': rbar, 'k_eff': keff, 'предел 1/ρ̄': 1 / rbar if rbar and rbar > 0 else np.nan, 'Judge': float(pd.to_numeric(d['Judge_score'], errors='coerce').mean())})
K = pd.DataFrame(rows).sort_values('k').set_index('группа')
show(K, 'Эффективное число голосов внутри каждой группы')
print()
print('Ключевая колонка — k_eff. Если при росте k с 3 до 10 значение k_eff')
print('растёт медленно и упирается в потолок, механизм из B9 подтверждён')
print('на независимых данных.')



Эффективное число голосов внутри каждой группы
------------------------------------------------------------
                                 k  ρ̄ (по ошибкам R1)  k_eff  предел 1/ρ̄  Judge
группа                                                                           
k=3 (референс)                   3              0.4876 1.5189       2.0510 0.8194
delphi_experts_group2_4models    4              0.6680 1.3315       1.4970 0.8071
delphi_experts_group3_5models    5              0.5691 1.5261       1.7572 0.8276
delphi_experts_group4_5models    5              0.4877 1.6944       2.0504 0.8169
delphi_experts_group6_5models    5              0.7378 1.2655       1.3555 0.6384
delphi_experts_group7_6models    6              0.5679 1.5626       1.7607 0.8249
delphi_experts_group5_7models    7              0.6675 1.3987       1.4982 0.8390
delphi_experts_group1_10models  10              0.5777 1.6131       1.7311 0.8198

Ключевая колонка — k_eff. Если при росте k с 3 до 10 значение k_eff
ра

In [6]:
KK = K.join(G[['ср. сила участников']])
rows = []
for pred in ['k', 'k_eff', 'ρ̄ (по ошибкам R1)']:
    sub = KK.dropna(subset=[pred, 'Judge'])
    r, p = stats.spearmanr(sub[pred], sub['Judge'])
    rows.append({'предиктор': pred, 'n групп': len(sub), 'rho со Judge': r, 'p': p})
show(pd.DataFrame(rows).set_index('предиктор'), 'Что лучше предсказывает качество: размер или эффективное число голосов')
print()
print('ПРЕДУПРЕЖДЕНИЕ О МОЩНОСТИ: при n = 8 группах ранговая корреляция должна')
print('превышать примерно 0.71, чтобы быть значимой на уровне 0.05. Поэтому')
print('незначимость здесь почти ничего не доказывает; смотреть надо на')
print('величину и знак.')



Что лучше предсказывает качество: размер или эффективное число голосов


----------------------------------------------------------------------
                    n групп  rho со Judge      p
предиктор                                       
k                         8        0.5367 0.1702
k_eff                     8        0.3095 0.4556
ρ̄ (по ошибкам R1)        8       -0.2619 0.5309

ПРЕДУПРЕЖДЕНИЕ О МОЩНОСТИ: при n = 8 группах ранговая корреляция должна
превышать примерно 0.71, чтобы быть значимой на уровне 0.05. Поэтому
незначимость здесь почти ничего не доказывает; смотреть надо на
величину и знак.


In [7]:
def tost(d, margin=0.05):
    n = len(d)
    m, se = (float(d.mean()), se_clt(d))
    t1 = (m - -margin) / se
    t2 = (m - margin) / se
    p1 = float(stats.t.sf(t1, n - 1))
    p2 = float(stats.t.cdf(t2, n - 1))
    return (max(p1, p2), m, se)
rows = []
for c in S.columns:
    if c == 'k=3 (референс)':
        continue
    d = S[c] - ref
    for margin in (0.05, 0.1):
        p, m, se = tost(d, margin)
        rows.append({'группа': c, 'k': KS[c], 'граница ±': margin, 'Δ': m, 'p TOST': p, 'вывод': 'эквивалентно k=3' if p < 0.05 else 'эквивалентность НЕ доказана'})
T = pd.DataFrame(rows).sort_values(['граница ±', 'k'])
show(T.set_index(['граница ±', 'группа']), 'Тесты эквивалентности против k = 3')



Тесты эквивалентности против k = 3
------------------------------------------------------------
                                           k       Δ  p TOST                        вывод
граница ± группа                                                                         
0.0500    delphi_experts_group2_4models    4 -0.0123  0.0120             эквивалентно k=3
          delphi_experts_group3_5models    5  0.0082  0.0026             эквивалентно k=3
          delphi_experts_group4_5models    5 -0.0025  0.0007             эквивалентно k=3
          delphi_experts_group6_5models    5 -0.1810  1.0000  эквивалентность НЕ доказана
          delphi_experts_group7_6models    6  0.0055  0.0011             эквивалентно k=3
          delphi_experts_group5_7models    7  0.0197  0.0211             эквивалентно k=3
          delphi_experts_group1_10models  10  0.0004  0.0010             эквивалентно k=3
0.1000    delphi_experts_group2_4models    4 -0.0123  0.0000             эквивалентно k=3
   

In [8]:
rows = []
for name, d in ALL.items():
    tot = [c for c in d.columns if re.match('r[123]_total_tokens__', c)]
    org = [c for c in ['organizer_after_r1_total_tokens', 'organizer_after_r2_total_tokens', 'final_synthesis_total_tokens'] if c in d.columns]
    gen = d[tot].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1).mean() if tot else np.nan
    agg = d[org].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1).mean() if org else np.nan
    q = float(pd.to_numeric(d['Judge_score'], errors='coerce').mean())
    rows.append({'группа': name, 'k': KS[name], 'S генерация': gen, 'V агрегация': agg, 'total': gen + agg, 'Judge': q, 'Judge на 10k токенов': q / ((gen + agg) / 10000.0) if gen + agg else np.nan})
CST = pd.DataFrame(rows).sort_values('k').set_index('группа')
show(CST, 'Стоимость и эффективность по размеру панели')



Стоимость и эффективность по размеру панели
------------------------------------------------------------
                                 k  S генерация  V агрегация      total  Judge  Judge на 10k токенов
группа                                                                                              
k=3 (референс)                   3   14456.2667   11596.0167 26052.2833 0.8194                0.3145
delphi_experts_group2_4models    4   21195.8000   14077.1000 35272.9000 0.8071                0.2288
delphi_experts_group3_5models    5   24587.4333   15472.7667 40060.2000 0.8276                0.2066
delphi_experts_group4_5models    5   36159.8833   16342.8000 52502.6833 0.8169                0.1556
delphi_experts_group6_5models    5   18549.4833   11801.6667 30351.1500 0.6384                0.2103
delphi_experts_group7_6models    6   38833.2333   17413.0333 56246.2667 0.8249                0.1467
delphi_experts_group5_7models    7   46695.8333   20670.7167 67366.5500 0.8390        

In [9]:
save(G.reset_index(), 'panel_size_scaling_groups.csv')
save(CMP.reset_index(), 'panel_size_scaling_vs_k3.csv')
save(K.reset_index(), 'panel_size_scaling_keff.csv')
save(T, 'panel_size_scaling_tost.csv')
save(CST.reset_index(), 'panel_size_scaling_cost.csv')


сохранено: out/panel_size_scaling_groups.csv  (8 строк)
сохранено: out/panel_size_scaling_vs_k3.csv  (7 строк)
сохранено: out/panel_size_scaling_keff.csv  (8 строк)
сохранено: out/panel_size_scaling_tost.csv  (14 строк)
сохранено: out/panel_size_scaling_cost.csv  (8 строк)


'\\\\?\\D:\\research_llmfmed\\research_llmfmed\\1design_alpha\\dataverse_gold\\github_repository\\analyses\\protocol_design\\results\\panel_size_scaling_cost.csv'